<a href="https://colab.research.google.com/github/Asritha5486/Image-Deblurring-Face-Identification/blob/main/Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install facenet-pytorch

  Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
  Using cached pillow-10.2.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (9.7 kB)
  Using cached torch-2.2.2-cp312-cp312-manylinux1_x86_64.whl.metadata (25 kB)
  Using cached torchvision-0.17.2-cp312-cp312-manylinux1_x86_64.whl.metadata (6.6 kB)
  Using cached nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_cupti_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cudnn_cu12-8.9.2.26-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cublas_cu12-12.1.3.1-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cufft_cu12-11.0.2.54-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_curand_cu12-10.3.2.106-py3-none-manylinux1_x86_64.whl.metada

In [ ]:
pip install --upgrade --force-reinstall pillow torchvision

  Using cached pillow-12.0.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (8.8 kB)
  Using cached torchvision-0.24.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (5.9 kB)
  Using cached numpy-2.3.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (62 kB)
  Using cached torch-2.9.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (30 kB)
  Using cached filelock-3.20.0-py3-none-any.whl.metadata (2.1 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached setuptools-80.9.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.5-py3-none-any.whl.metadata (6.3 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached fsspec-2025.10.0-py3-none-any.whl.metadata (10 kB)
  Using cached nvidia_cuda_nvrtc_cu12-12.8.93-py3-none-manylinux2010_x86_64.manylinux_2_12_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cuda_runtime

In [ ]:
# ===============================
# 🔁 Full Pipeline: Deblur + Detect + Recognize
# ===============================

# ===============================
# 1️⃣ Imports
# ===============================
import torch
import torch.nn as nn
from PIL import Image
from torchvision import transforms
import matplotlib.pyplot as plt
from facenet_pytorch import InceptionResnetV1, MTCNN
import numpy as np
import cv2
import pickle
from sklearn.metrics.pairwise import cosine_similarity

# ===============================
# 2️⃣ DeblurGAN Generator Definition
# ===============================
class DeblurGANGenerator(nn.Module):
    def __init__(self):
        super(DeblurGANGenerator, self).__init__()
        self.model = nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=64, kernel_size=9, stride=1, padding=4),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels=64, out_channels=3, kernel_size=9, stride=1, padding=4),
            nn.Tanh()
        )

    def forward(self, x):
        return self.model(x)

# ===============================
# 3️⃣ Device Setup
# ===============================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Using device: {device}")

# ===============================
# 4️⃣ Load Deblurring Model
# ===============================
G = DeblurGANGenerator().to(device)
checkpoint = torch.load("/content/generator.pth", map_location=device, weights_only=True)
new_state_dict = {k.replace("main.", "model."): v for k, v in checkpoint.items()}
G.load_state_dict(new_state_dict, strict=False)
G.eval()
print("✅ DeblurGAN Generator loaded successfully.")

# ===============================
# 5️⃣ Deblur Function
# ===============================
def deblur_image(input_path, output_path):
    blur_img = Image.open(input_path).convert("RGB")
    transform = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.ToTensor(),
    ])
    input_tensor = transform(blur_img).unsqueeze(0).to(device)

    with torch.no_grad():
        deblurred = G(input_tensor)

    deblurred_img_np = (
        deblurred.squeeze(0).cpu().permute(1, 2, 0).clamp(0, 1).numpy()
    )
    deblurred_pil = Image.fromarray((deblurred_img_np * 255).astype('uint8'))
    deblurred_pil.save(output_path)
    print(f"✅ Deblurred image saved at: {output_path}")
    return output_path

# ===============================
# 6️⃣ Face Recognition Setup
# ===============================
with open("known_face.pkl", "rb") as f:
    data = pickle.load(f)
known_embs = data["embeddings"]
known_labels = data["labels"]
known_embs = known_embs / np.linalg.norm(known_embs, axis=1, keepdims=True)
print(f"✅ Loaded {len(known_embs)} known embeddings.")

facenet_model = InceptionResnetV1(pretrained='vggface2').eval().to(device)
mtcnn = MTCNN(image_size=160, margin=10, keep_all=True, device=device)

# ===============================
# 7️⃣ Face Recognition Function
# ===============================
def recognize_faces(img_path, output_path="recognized_output.png", threshold=0.7):
    img = Image.open(img_path).convert("RGB")
    img_np = np.array(img)
    boxes, _ = mtcnn.detect(img)

    if boxes is None:
        print("⚠️ No faces detected in the image.")
        return None

    for i, box in enumerate(boxes):
        x1, y1, x2, y2 = map(int, box)
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(img_np.shape[1], x2), min(img_np.shape[0], y2)

        face_list = mtcnn.extract(img, [box], save_path=None)
        if face_list is None or len(face_list) == 0:
            print(f"Face {i+1}: ❌ Could not extract face")
            continue

        face_tensor = face_list[0].unsqueeze(0).to(device)

        with torch.no_grad():
            embedding = facenet_model(face_tensor).cpu().numpy()
            embedding /= np.linalg.norm(embedding)

        sims = cosine_similarity(embedding, known_embs)[0]
        best_idx = np.argmax(sims)
        best_score = sims[best_idx]
        best_label = known_labels[best_idx]

        if best_score > threshold:
            print(f"Face {i+1}: ✅ Known Person Detected: {best_label} (similarity: {best_score:.2f})")
            color = (0, 255, 0)
            label_text = best_label
        else:
            print(f"Face {i+1}: ❌ Unknown Person (similarity: {best_score:.2f})")
            color = (0, 0, 255)
            label_text = "Unknown"

        cv2.rectangle(img_np, (x1, y1), (x2, y2), color, 2)
        cv2.putText(img_np, label_text, (x1, y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)

    cv2.imwrite(output_path, cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR))
    print(f"✅ Recognition output saved at: {output_path}")
    return output_path

# ===============================
# 8️⃣ Combined Pipeline Function
# ===============================
def full_pipeline(input_blur_path):
    print("\n🚀 Starting Full Pipeline...")

    # Step 1: Deblur
    deblur_path = "temp_deblurred.png"
    deblur_image(input_blur_path, deblur_path)

    # Step 2: Detect + Recognize
    recognize_faces(deblur_path, "final_output.png")

    print("🎯 Pipeline Completed Successfully.")
    return "final_output.png"

# ===============================
# 9️⃣ Run the Pipeline
# ===============================
input_image_path = "/content/c_blurred.png"
final_output = full_pipeline(input_image_path)

# Optional: Display
plt.imshow(Image.open(final_output))
plt.axis("off")
plt.title("Final Output (Deblurred + Recognized)")
plt.show()


ImportError: cannot import name 'is_directory' from 'PIL._util' (/usr/local/lib/python3.12/dist-packages/PIL/_util.py)

MODEL EVALUATION

In [ ]:
# ==============================
# 📊 5.8 Model Evaluation Script
# ==============================

import os
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import cv2
from tqdm import tqdm
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
)
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr

# ==============================
# 1️⃣ Paths
# ==============================
BLUR_DIR = "/content/blurred_images"
GT_DIR = "/content/sharp_images"
DEBLUR_DIR = "/content/eval_deblurred"
LABELS_CSV = "/content/labels.csv"

os.makedirs(DEBLUR_DIR, exist_ok=True)

# ==============================
# 2️⃣ Deblur Evaluation (PSNR / SSIM)
# ==============================
psnr_list, ssim_list, files_evaluated = [], [], []

for fname in tqdm(sorted(os.listdir(BLUR_DIR)), desc="Evaluating Deblurring"):
    blur_path = os.path.join(BLUR_DIR, fname)
    gt_path = os.path.join(GT_DIR, fname)
    if not os.path.exists(gt_path):
        continue

    # Deblur using your pipeline function
    out_path = os.path.join(DEBLUR_DIR, fname)
    deblur_image(blur_path, out_path)

    # Read images
    gt_img = np.array(Image.open(gt_path).convert("RGB")).astype(np.float32) / 255.0
    deblur_img = np.array(Image.open(out_path).convert("RGB")).astype(np.float32) / 255.0

    # Resize if mismatch
    if deblur_img.shape != gt_img.shape:
        deblur_img = cv2.resize(deblur_img, (gt_img.shape[1], gt_img.shape[0]))

    psnr_val = psnr(gt_img, deblur_img, data_range=1.0)
    ssim_val = ssim(gt_img, deblur_img, channel_axis=-1, data_range=1.0)
    psnr_list.append(psnr_val)
    ssim_list.append(ssim_val)
    files_evaluated.append(fname)

print(f"\n🧮 Mean PSNR: {np.mean(psnr_list):.2f} | Mean SSIM: {np.mean(ssim_list):.4f}")

# ==============================
# 3️⃣ Recognition Evaluation
# ==============================
labels_df = pd.read_csv(LABELS_CSV)
labels_df = labels_df.set_index("filename")

y_true, y_pred = [], []

for fname in tqdm(sorted(os.listdir(DEBLUR_DIR)), desc="Evaluating Recognition"):
    if fname not in labels_df.index:
        continue
    true_label = labels_df.loc[fname, "true_label"]

    out_recog = f"/content/temp_recog.png"
    recognize_faces(os.path.join(DEBLUR_DIR, fname), output_path=out_recog, threshold=0.7)

    # Capture printed result (simplified: we re-run recognition in a silent mode)
    img = Image.open(os.path.join(DEBLUR_DIR, fname)).convert("RGB")
    boxes, _ = mtcnn.detect(img)
    if boxes is None:
        pred = "NoFace"
    else:
        face = mtcnn.extract(img, [boxes[0]], save_path=None)[0].unsqueeze(0).to(device)
        with torch.no_grad():
            emb = facenet_model(face).cpu().numpy()
            emb /= np.linalg.norm(emb)
        sims = cosine_similarity(emb, known_embs)[0]
        best_idx = np.argmax(sims)
        best_score = sims[best_idx]
        pred = known_labels[best_idx] if best_score > 0.7 else "Unknown"

    y_true.append(true_label)
    y_pred.append(pred)

# Metrics
acc = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred, average="macro", zero_division=0)
rec = recall_score(y_true, y_pred, average="macro", zero_division=0)
f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)

print(f"\n🎯 Recognition Metrics:\nAccuracy={acc:.3f}, Precision={prec:.3f}, Recall={rec:.3f}, F1={f1:.3f}")

# Save results table
table_df = pd.DataFrame({
    "PSNR_mean": [np.mean(psnr_list)],
    "SSIM_mean": [np.mean(ssim_list)],
    "Accuracy": [acc],
    "Precision": [prec],
    "Recall": [rec],
    "F1_score": [f1]
})
table_df.to_csv("Table_5_1_metrics.csv", index=False)
print("📊 Saved: Table_5_1_metrics.csv")

# ==============================
# 4️⃣ Qualitative: Figure 5.3
# ==============================
sample_files = sorted(files_evaluated)[:5]
fig, axes = plt.subplots(len(sample_files), 3, figsize=(12, 5*len(sample_files)))

for i, fname in enumerate(sample_files):
    blur_img = Image.open(os.path.join(BLUR_DIR, fname)).convert("RGB")
    deblur_img = Image.open(os.path.join(DEBLUR_DIR, fname)).convert("RGB")
    final_img = Image.open("final_output.png").convert("RGB") if os.path.exists("final_output.png") else deblur_img

    axes[i,0].imshow(blur_img); axes[i,0].axis("off"); axes[i,0].set_title("Blurred")
    axes[i,1].imshow(deblur_img); axes[i,1].axis("off"); axes[i,1].set_title("Deblurred")
    axes[i,2].imshow(final_img); axes[i,2].axis("off"); axes[i,2].set_title("Recognition Result")

plt.tight_layout()
plt.savefig("Figure_5_3_results.png", dpi=300)
print("🖼 Saved: Figure_5_3_results.png")

print("\n✅ Evaluation Completed Successfully!")


In [ ]:
!pip install gradio

In [ ]:
import gradio as gr
import torch
import torch.nn as nn
from PIL import Image
from torchvision import transforms
import numpy as np
import cv2
from facenet_pytorch import InceptionResnetV1, MTCNN
from sklearn.metrics.pairwise import cosine_similarity
import pickle

# ==========================
# 1️⃣ Load DeblurGAN Model
# ==========================
class DeblurGANGenerator(nn.Module):
    def __init__(self):
        super(DeblurGANGenerator, self).__init__()
        self.model = nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=64, kernel_size=9, stride=1, padding=4),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels=64, out_channels=3, kernel_size=9, stride=1, padding=4),
            nn.Tanh()
        )

    def forward(self, x):
        return self.model(x)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("✅ Using device:", device)

G = DeblurGANGenerator().to(device)
checkpoint = torch.load("/content/generator.pth", map_location=device)
new_state_dict = {k.replace("main.", "model."): v for k, v in checkpoint.items()}
G.load_state_dict(new_state_dict, strict=False)
G.eval()

# ==========================
# 2️⃣ Load Face Recognition Models
# ==========================
with open("/content/known_face.pkl", "rb") as f:
    data = pickle.load(f)
known_embs = data["embeddings"]
known_labels = data["labels"]
known_embs = known_embs / np.linalg.norm(known_embs, axis=1, keepdims=True)

facenet_model = InceptionResnetV1(pretrained='vggface2').eval().to(device)
mtcnn = MTCNN(image_size=160, margin=10, keep_all=True, device=device)

# ==========================
# 3️⃣ Define Processing Pipeline
# ==========================
def process_image(image):
    if image is None:
        return None, "⚠️ No image uploaded"

    img = image.convert("RGB")

    # Step 1: Deblur
    transform = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.ToTensor()
    ])
    input_tensor = transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        deblurred = G(input_tensor)
    deblurred_img_np = (
        deblurred.squeeze(0).cpu().permute(1, 2, 0).clamp(0, 1).numpy()
    )
    deblurred_img = (deblurred_img_np * 255).astype('uint8')
    deblurred_pil = Image.fromarray(deblurred_img)

    # Step 2: Face Detection + Recognition
    img_np = np.array(deblurred_pil)
    boxes, _ = mtcnn.detect(deblurred_pil)

    if boxes is None:
        return deblurred_pil, "⚠️ No faces detected"

    for i, box in enumerate(boxes):
        x1, y1, x2, y2 = map(int, box)
        face_list = mtcnn.extract(deblurred_pil, [box], save_path=None)
        if face_list is None or len(face_list) == 0:
            continue

        face_tensor = face_list[0].unsqueeze(0).to(device)
        with torch.no_grad():
            emb = facenet_model(face_tensor).cpu().numpy()
            emb /= np.linalg.norm(emb)

        sims = cosine_similarity(emb, known_embs)[0]
        best_idx = np.argmax(sims)
        best_score = sims[best_idx]
        best_label = known_labels[best_idx] if best_score > 0.7 else "Unknown"

        color = (0, 255, 0) if best_score > 0.7 else (255, 0, 0)
        label_text = f"{best_label} ({best_score:.2f})" if best_score > 0.7 else "Unknown"

        cv2.rectangle(img_np, (x1, y1), (x2, y2), color, 2)
        cv2.putText(img_np, label_text, (x1, y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)

    result_img = Image.fromarray(img_np)
    return result_img, "✅ Process completed successfully"

# ==========================
# 4️⃣ Launch Gradio Interface
# ==========================
interface = gr.Interface(
    fn=process_image,
    inputs=gr.Image(type="pil", label="Upload Blurred Image"),
    outputs=[
        gr.Image(type="pil", label="Output (Deblurred + Recognized)"),
        gr.Textbox(label="Status")
    ],
    title="🧠 Deblur + Face Recognition Pipeline",
    description="Upload a blurred image — it will be deblurred and checked against known thieves."
)

interface.launch(debug=True)


In [ ]:
!pip install --upgrade torch torchvision Pillow==10.3.0 --quiet

In [ ]:
import gradio as gr
import torch
import torch.nn as nn
from PIL import Image
from torchvision import transforms
import numpy as np
import cv2
from facenet_pytorch import InceptionResnetV1, MTCNN
from sklearn.metrics.pairwise import cosine_similarity
import pickle

# ==========================
# 1️⃣ Load DeblurGAN Model
# ==========================
class DeblurGANGenerator(nn.Module):
    def __init__(self):
        super(DeblurGANGenerator, self).__init__()
        self.model = nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=64, kernel_size=9, stride=1, padding=4),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels=64, out_channels=3, kernel_size=9, stride=1, padding=4),
            nn.Tanh()
        )

    def forward(self, x):
        return self.model(x)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("✅ Using device:", device)

G = DeblurGANGenerator().to(device)
checkpoint = torch.load("/content/generator.pth", map_location=device)
new_state_dict = {k.replace("main.", "model."): v for k, v in checkpoint.items()}
G.load_state_dict(new_state_dict, strict=False)
G.eval()

# ==========================
# 2️⃣ Load Face Recognition Models
# ==========================
with open("/content/known_face.pkl", "rb") as f:
    data = pickle.load(f)
known_embs = data["embeddings"]
known_labels = data["labels"]
known_embs = known_embs / np.linalg.norm(known_embs, axis=1, keepdims=True)

facenet_model = InceptionResnetV1(pretrained='vggface2').eval().to(device)
mtcnn = MTCNN(image_size=160, margin=10, keep_all=True, device=device)

# ==========================
# 3️⃣ Define Processing Pipeline
# ==========================
def process_image(image):
    if image is None:
        return None, "⚠️ No image uploaded"

    img = image.convert("RGB")

    # Step 1: Deblur
    transform = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.ToTensor()
    ])
    input_tensor = transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        deblurred = G(input_tensor)
    deblurred_img_np = (
        deblurred.squeeze(0).cpu().permute(1, 2, 0).clamp(0, 1).numpy()
    )
    deblurred_img = (deblurred_img_np * 255).astype('uint8')
    deblurred_pil = Image.fromarray(deblurred_img)

    # Step 2: Face Detection + Recognition
    img_np = np.array(deblurred_pil)
    boxes, _ = mtcnn.detect(deblurred_pil)

    if boxes is None:
        return deblurred_pil, "⚠️ No faces detected"

    for i, box in enumerate(boxes):
        x1, y1, x2, y2 = map(int, box)
        face_list = mtcnn.extract(deblurred_pil, [box], save_path=None)
        if face_list is None or len(face_list) == 0:
            continue

        face_tensor = face_list[0].unsqueeze(0).to(device)
        with torch.no_grad():
            emb = facenet_model(face_tensor).cpu().numpy()
            emb /= np.linalg.norm(emb)

        sims = cosine_similarity(emb, known_embs)[0]
        best_idx = np.argmax(sims)
        best_score = sims[best_idx]
        best_label = known_labels[best_idx] if best_score > 0.7 else "Unknown"

        color = (0, 255, 0) if best_score > 0.7 else (255, 0, 0)
        label_text = f"{best_label} ({best_score:.2f})" if best_score > 0.7 else "Unknown"

        cv2.rectangle(img_np, (x1, y1), (x2, y2), color, 2)
        cv2.putText(img_np, label_text, (x1, y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)

    result_img = Image.fromarray(img_np)
    return result_img, "✅ Process completed successfully"

# ==========================
# 4️⃣ Launch Gradio Interface
# ==========================
interface = gr.Interface(
    fn=process_image,
    inputs=gr.Image(type="pil", label="Upload Blurred Image"),
    outputs=[
        gr.Image(type="pil", label="Output (Deblurred + Recognized)"),
        gr.Textbox(label="Status")
    ],
    title="Deblurring and Face Recognition System",
    description="Upload a blurred image — it will be deblurred and checked against known thieves."
)

interface.launch(debug=True)


ImportError: cannot import name 'is_directory' from 'PIL._util' (/usr/local/lib/python3.12/dist-packages/PIL/_util.py)